# 03 — Model Evaluation

Evaluación avanzada del modelo de churn con enfoque técnico y de negocio.

## Objetivos
- Cargar el pipeline entrenado.
- Evaluar Accuracy, Precision, Recall, F1 y ROC-AUC.
- Analizar matriz de confusión.
- Construir curvas ROC y Precision-Recall.
- Comparar distintos umbrales de clasificación.
- Analizar falsos positivos y falsos negativos.
- Interpretar la importancia de variables.
- Guardar métricas y gráficos en `reports/`.


## 1. Configuración del entorno

Ejecute este notebook desde la raíz del repositorio. En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


## 2. Importación de librerías


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from src.data import load_customer_data
from src.features import split_features_target
from src.models.evaluation import evaluate_binary_classifier
from src.utils import get_project_path, get_logger
logger = get_logger("notebook.model_evaluation")


## 3. Carga de datos y modelo


In [ ]:
DATA_PATH = get_project_path("data", "raw", "customer_churn.csv")
MODEL_PATH = get_project_path("artifacts", "models", "selected_churn_pipeline.joblib")
df = load_customer_data(DATA_PATH)
if not MODEL_PATH.exists():
    raise FileNotFoundError("Ejecute primero 02_model_development.ipynb")
pipeline = joblib.load(MODEL_PATH)
print(df.shape, MODEL_PATH)


## 4. Reconstrucción del conjunto de prueba


In [ ]:
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(len(X_train), len(X_test), y_test.mean())


## 5. Predicciones y probabilidades


In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
evaluation_df = X_test.copy()
evaluation_df["actual_churn"] = y_test.values
evaluation_df["predicted_churn"] = y_pred
evaluation_df["churn_probability"] = y_proba
evaluation_df.head()


## 6. Métricas principales


In [ ]:
metrics = evaluate_binary_classifier(y_test, y_pred, y_proba)
pd.Series(metrics, name="value").to_frame()


In [ ]:
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], digits=4))


### Interpretación
- **Precision:** de los clientes predichos como churn, cuántos realmente cancelan.
- **Recall:** de los clientes que realmente cancelan, cuántos fueron detectados.
- **F1-score:** equilibrio entre Precision y Recall.
- **ROC-AUC:** capacidad de ordenar correctamente clientes de mayor y menor riesgo.


## 7. Matriz de confusión


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm


In [ ]:
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No churn", "Churn"])
display_cm.plot(values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
tn, fp, fn, tp = cm.ravel()
pd.DataFrame({"metric":["TN","FP","FN","TP"],"count":[tn,fp,fn,tp]})


### Discusión de negocio
- **Falso positivo:** campaña innecesaria.
- **Falso negativo:** cliente perdido sin intervención.
En retención, normalmente el falso negativo tiene mayor costo.


## 8. Curva ROC


In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],linestyle="--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend(); plt.grid(alpha=.3); plt.show()


## 9. Curva Precision-Recall


In [ ]:
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)
average_precision = average_precision_score(y_test, y_proba)
plt.figure(figsize=(8,6))
plt.plot(recall_curve, precision_curve, label=f"AP = {average_precision:.3f}")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Baseline = {y_test.mean():.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(alpha=.3); plt.show()


## 10. Evaluación de umbrales


In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
rows=[]
for threshold in thresholds:
    pred=(y_proba>=threshold).astype(int)
    rows.append({"threshold":threshold,"accuracy":accuracy_score(y_test,pred),"precision":precision_score(y_test,pred,zero_division=0),"recall":recall_score(y_test,pred,zero_division=0),"f1":f1_score(y_test,pred,zero_division=0),"predicted_churn_rate":pred.mean()})
threshold_df=pd.DataFrame(rows)
threshold_df


In [ ]:
ax=threshold_df.plot(x="threshold",y=["precision","recall","f1"],figsize=(10,6),marker="o")
ax.set_ylim(0,1); ax.grid(alpha=.3); plt.show()


## 11. Selección de umbral por criterio


In [ ]:
best_f1_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_recall_row = threshold_df.query("precision >= 0.40").sort_values("recall",ascending=False).head(1)
display(best_f1_row.to_frame().T)
display(best_recall_row)


### Regla pedagógica sugerida
Use umbral bajo si perder clientes es muy costoso; umbral alto si la campaña es costosa. Seleccione con costos y capacidad operativa.


## 12. Simulación simple de costos


In [ ]:
RETENTION_CAMPAIGN_COST=25
FALSE_NEGATIVE_COST=300
TRUE_POSITIVE_BENEFIT=180
cost_rows=[]
for threshold in thresholds:
    pred=(y_proba>=threshold).astype(int)
    tn2,fp2,fn2,tp2=confusion_matrix(y_test,pred).ravel()
    net=tp2*TRUE_POSITIVE_BENEFIT-(tp2+fp2)*RETENTION_CAMPAIGN_COST-fn2*FALSE_NEGATIVE_COST
    cost_rows.append({"threshold":threshold,"tp":tp2,"fp":fp2,"fn":fn2,"net_value":net})
cost_df=pd.DataFrame(cost_rows)
cost_df.sort_values("net_value",ascending=False).head()


In [ ]:
ax=cost_df.plot(x="threshold",y="net_value",figsize=(9,5),marker="o",legend=False)
ax.grid(alpha=.3); plt.show()


## 13. Análisis de falsos positivos


In [ ]:
false_positives=evaluation_df.query("actual_churn == 0 and predicted_churn == 1").sort_values("churn_probability",ascending=False)
print(len(false_positives))
false_positives.head(10)


## 14. Análisis de falsos negativos


In [ ]:
false_negatives=evaluation_df.query("actual_churn == 1 and predicted_churn == 0").sort_values("churn_probability")
print(len(false_negatives))
false_negatives.head(10)


## 15. Comparación de perfiles


In [ ]:
profile_columns=["tenure_months","monthly_fee","support_calls","complaints","last_payment_delay","digital_usage_score","marketing_score"]
profile_comparison=pd.DataFrame({"true_positive":evaluation_df.query("actual_churn == 1 and predicted_churn == 1")[profile_columns].mean(),"false_negative":false_negatives[profile_columns].mean(),"false_positive":false_positives[profile_columns].mean(),"true_negative":evaluation_df.query("actual_churn == 0 and predicted_churn == 0")[profile_columns].mean()})
profile_comparison


## 16. Importancia de variables


In [ ]:
classifier=pipeline.named_steps["classifier"]
preprocessor=pipeline.named_steps["preprocessor"]
feature_names=preprocessor.get_feature_names_out()
importance_df=None
if hasattr(classifier,"feature_importances_"):
    importance_df=pd.DataFrame({"feature":feature_names,"importance":classifier.feature_importances_}).sort_values("importance",ascending=False)
elif hasattr(classifier,"coef_"):
    importance_df=pd.DataFrame({"feature":feature_names,"coefficient":classifier.coef_[0]}).assign(absolute_importance=lambda x:x["coefficient"].abs()).sort_values("absolute_importance",ascending=False)
importance_df.head(15)


In [ ]:
if "importance" in importance_df.columns:
    plot_data=importance_df.head(15).sort_values("importance"); x_col="importance"
else:
    plot_data=importance_df.head(15).sort_values("absolute_importance"); x_col="absolute_importance"
plot_data.plot(x="feature",y=x_col,kind="barh",figsize=(10,7),legend=False)
plt.title("Top Model Features"); plt.tight_layout(); plt.show()


## 17. Guardado de métricas y reportes


In [ ]:
METRICS_OUTPUT=get_project_path("reports","metrics","advanced_evaluation.json",create_parent=True)
THRESHOLD_OUTPUT=get_project_path("reports","metrics","threshold_analysis.csv",create_parent=True)
ERRORS_OUTPUT=get_project_path("reports","predictions","evaluation_errors.csv",create_parent=True)
payload={"base_metrics":metrics,"roc_auc":roc_auc,"average_precision":average_precision,"confusion_matrix":{"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)},"best_threshold_by_f1":float(best_f1_row["threshold"]),"best_f1":float(best_f1_row["f1"])}
METRICS_OUTPUT.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
threshold_df.to_csv(THRESHOLD_OUTPUT,index=False)
pd.concat([false_positives.assign(error_type="false_positive"),false_negatives.assign(error_type="false_negative")]).to_csv(ERRORS_OUTPUT,index=False)
print(METRICS_OUTPUT, THRESHOLD_OUTPUT, ERRORS_OUTPUT)


## 18. Exportación de gráficos


In [ ]:
FIGURES_DIR=get_project_path("artifacts","figures","placeholder.txt",create_parent=True).parent
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0,1],[0,1],linestyle="--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR/"roc_curve.png",dpi=150)
plt.close()
print(FIGURES_DIR/"roc_curve.png")


## 19. Preguntas para estudiantes
1. ¿Por qué 0.50 no siempre es óptimo?
2. ¿Qué error es más costoso para NovaTel?
3. ¿Qué métrica usaría para priorizar clientes?
4. ¿Cómo cambia la decisión si la campaña cuesta S/100?
5. ¿Qué perfiles predominan entre falsos negativos?
6. ¿Por qué importancia no implica causalidad?


## 20. Checklist de cierre
- [ ] Métricas base calculadas.
- [ ] Matriz de confusión interpretada.
- [ ] Curvas ROC y PR creadas.
- [ ] Umbrales evaluados.
- [ ] Costos simulados.
- [ ] FP y FN analizados.
- [ ] Importancia revisada.
- [ ] Reportes guardados.


## Resultado esperado
La evaluación se convierte en una decisión reproducible de negocio, no solo en una lectura de Accuracy.
